# Colab Part A: hybrid TimesFM + LSTM heads on CSE, Binance and Yahoo

**J26-SE-325, Component 1.** The TAF architecture: a frozen TimesFM 2.5 base forecasts
returns, and an LSTM/MLP residual head corrects it from technical indicators, per-tier
sentiment, point-in-time fundamentals and the asset class. This notebook trains one head per
horizon (5 and 21 trading days) and exports them (~a few hundred KB) for the local service.

**Run Part D first** (`colab_fundamentals_backfill.ipynb`): the fundamentals inputs come from
its parquet. Without it the fundamentals columns are zero and `has_fundamentals = 0`.

**Runtime:** T4 GPU. **Resumable:** bars, features and each finished head are written to
Drive; after a disconnect run all cells again and finished stages are skipped.

(`colab_finetune.ipynb` remains for the optional LoRA adaptation of the base model itself.)

In [ ]:
# === BOOTSTRAP (same contract as colab_finetune.ipynb) ================================
# Clone or update the branch, chdir into the component, install PIP_PKGS. Safe to re-run.
REPO_URL = "https://github.com/SE-Y4S1/J26-SE-325.git"
BRANCH = "Nivakaran"  # the component is NOT on main
SUBDIR = "backend/Portfolio-Optimization"
PIP_PKGS = "timesfm[torch]==2.0.2 pandas-ta-openbb yfinance pypdf pyyaml requests pydantic pydantic-settings"

import importlib, os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
OWN_PACKAGES = {"agent", "configs", "data", "evaluation", "experiments", "features",
                "forecasting", "fundamentals", "news", "optimization", "decision", "service"}


def find_root(start):
    for base in (start, *start.parents):
        if (base / "configs/universe.yaml").exists() and (base / "fundamentals").exists():
            return base
    return None


root = find_root(Path.cwd())
if root is None:
    if not IN_COLAB:
        raise SystemExit("cd into backend/Portfolio-Optimization first.")
    checkout = Path("/content/J26-SE-325")
    if (checkout / SUBDIR).exists():
        if subprocess.run(["git", "-C", str(checkout), "fetch", "--depth", "1", "origin", BRANCH]).returncode == 0:
            subprocess.run(["git", "-C", str(checkout), "reset", "--hard", "FETCH_HEAD"])
    else:
        if subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)]).returncode:
            from getpass import getpass
            token = getpass("Clone failed. GitHub token (repo scope), blank to abort: ").strip()
            if not token:
                raise SystemExit("Cannot reach " + REPO_URL)
            subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                            REPO_URL.replace("https://", "https://" + token + "@"), str(checkout)], check=True)
    root = checkout / SUBDIR

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
for _name in [n for n in sys.modules if n.split(".")[0] in OWN_PACKAGES]:
    del sys.modules[_name]
importlib.invalidate_caches()
if IN_COLAB and PIP_PKGS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PIP_PKGS.split(), check=True)
print("OK root=" + str(root))

In [ ]:
import torch

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"

# The adapter registry reads artifacts/env_report.json (gitignored), so write it here.
from forecasting.env_report import write_env_report

report = write_env_report()
assert report["optional_forecasters"]["timesfm"]["available"], report["optional_forecasters"]["timesfm"]

In [ ]:
from pathlib import Path

OUT = Path("artifacts/hybrid_training")
BACKFILL = None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/j26_hybrid")
    candidate = Path("/content/drive/MyDrive/j26_se_325_backfill/fundamentals_backfill.parquet")
    BACKFILL = candidate if candidate.exists() else None
OUT.mkdir(parents=True, exist_ok=True)
print("out:", OUT, "| fundamentals backfill:", BACKFILL or "NOT FOUND (fundamentals inputs will be zero)")

## Smoke run

Five CSE names, one horizon, one epoch: catches data, shape and device problems in minutes.

In [ ]:
from experiments.train_hybrid_heads import main

args = ["--out", str(OUT / "smoke"), "--cse-limit", "5", "--horizons", "5",
        "--epochs", "1", "--walk-forward-points", "20"]
if BACKFILL:
    args += ["--backfill", str(BACKFILL)]
main(args)

## Full training

Every CSE security with >= 200 sessions, the top 50 Binance USDT pairs, and the US universe.
The base forecasts (walk-forward TimesFM passes) dominate the run time.

In [ ]:
args = ["--out", str(OUT)]
if BACKFILL:
    args += ["--backfill", str(BACKFILL)]
main(args)

## Holdout result (RQ1 input)

Pinball loss on the time-held-out final 20% of dates: base TimesFM vs the hybrid, overall and
per asset class. Lower is better. A hybrid that does not beat its base on a class is reported
as such, not hidden.

In [ ]:
import json

import pandas as pd

rows = []
for head in sorted(OUT.glob("h*/meta.json")):
    meta = json.loads(head.read_text())
    m = meta["metrics"]
    for key in m:
        if key.startswith("pinball_base_"):
            cls = key.removeprefix("pinball_base_")
            rows.append({"horizon": meta["horizon"], "asset_class": cls, "base": m[key],
                         "hybrid": m.get("pinball_hybrid_" + cls), "rows": m.get("holdout_rows_" + cls)})
table = pd.DataFrame(rows)
table["improvement_%"] = 100 * (1 - table["hybrid"] / table["base"])
table

## Bring the heads home

Then, on the local machine:

```bash
python -m forecasting.serving --install hybrid_heads.zip
```

That registers `hybrid_h5` / `hybrid_h21` in the model registry, so responses carry a real
`model_version`. Serving also needs TimesFM's weights cached locally (~821 MB, once):
`huggingface-cli download google/timesfm-2.5-200m-pytorch`. Until then the analysis pipeline
uses a labelled empirical band and `/forecast` answers 503 with that reason.

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(str(OUT / "hybrid_heads.zip"))